# Lepton reconstruction efficiency vs $L_{xy}$, per gen lepton

Measure the reconstruction efficiency of electrons, photons, PF muons, and DSA muons as a function of the gen-level dark photon $L_{xy}$, counting each gen lepton separately. `leptonRecoEff_vs_lxy.ipynb` does the same per gen dark photon.

**How the efficiency is defined.** Every gen lepton from a dark photon ($A$) decay is one trial, so each $A \to \ell\ell$ contributes two entries to the denominator:
1. Gen leptons are the two direct daughters of each gen $A$ of the right flavor. Electrons and photons are compared to the electrons from $A \to ee$, PF and DSA muons to the muons from $A \to \mu\mu$.
2. Each gen lepton is given the $L_{xy}$ of its parent $A$ (its production vertex), so the x axis is the same as in the per-dark-photon notebook.
3. Gen leptons are matched **one-to-one** to reco objects of the given type within $\Delta R(\text{gen lepton}, \text{reco}) < 0.2$, closest pairs first. A single reco object near both leptons of a collimated pair therefore reconstructs only one of them. Only reco objects that pass the channel's object cuts can be matched.
4. $\epsilon(L_{xy}) = \dfrac{\text{number of matched gen leptons in the } L_{xy} \text{ bin}}{\text{number of gen leptons in the } L_{xy} \text{ bin}}$

There is no `min_n` here: each gen lepton is simply matched or not. No acceptance ($p_T$, $\eta$) requirement is placed on the gen leptons.

The numerator and denominator hists are defined at the end of `sidm/definitions/hists.py` (matching cone: `recoEff_lepton_dr`) and grouped in the `lepton_recoEff_perLepton_lxy` hist collection. Uncertainties are 68.3% Clopper-Pearson intervals.

In [ ]:
# python
import os
import sys
import importlib
# columnar analysis
from coffea import processor
import awkward as ak
import hist
from dask.distributed import Client, performance_report
# numerics
import numpy as np
from scipy import stats
# local
sidm_path = str(os.getcwd()).split("/sidm")[0]
if sidm_path not in sys.path: sys.path.insert(1, sidm_path)
from sidm.tools import utilities, sidm_processor, scaleout, llpnanoaodschema
# always reload local modules to pick up changes during development
importlib.reload(utilities)
importlib.reload(sidm_processor)
importlib.reload(scaleout)
importlib.reload(llpnanoaodschema)
# plotting
import matplotlib.pyplot as plt
utilities.set_plot_style()
%matplotlib inline

In [ ]:
client = scaleout.make_dask_client("tls://localhost:8786")
client

Run over every available lifetime of four (bound state mass, dark photon mass) points that more or less span the possible gen-lepton $\Delta R$ values. Each sample is plotted on its own; samples are never summed or mixed. Hists are filled unweighted (raw counts), so the binomial uncertainty is exact. Several channels are run at once (e.g. `base` vs `base_sr`, or `baseNoLjNoLjsource` vs `baseNoLj` to see how much of the inefficiency comes from reconstruction itself vs. the LJ-source object selection) and shown side by side.

In [ ]:
# all available lifetimes for each (bound state mass, dark photon mass) point
mass_points = {
    "100GeV_0p25GeV":  ["0p02mm", "0p2mm", "2p0mm", "10p0mm", "20p0mm"],
    "100GeV_5p0GeV":   ["0p4mm", "4p0mm", "40p0mm", "200mm", "400mm"],
    "1000GeV_0p25GeV": ["0p002mm", "0p02mm", "0p2mm", "1p0mm", "2p0mm"],
    "1000GeV_5p0GeV":  ["0p04mm", "0p4mm", "4p0mm", "20p0mm", "40p0mm"],
}
# every sample is processed and plotted separately
samples = [f"2Mu2E_{mp}_{ctau}" for mp, ctaus in mass_points.items() for ctau in ctaus]

max_files = -1
fileset = utilities.make_fileset(samples, "llpNanoAOD_v2", max_files=max_files, location_cfg="signal_2mu2e_v10.yaml")

runner = processor.Runner(
    #executor=processor.IterativeExecutor(),
    executor=processor.DaskExecutor(client=client),
    schema=llpnanoaodschema.LLPNanoAODSchema,
    #maxchunks=1,
    skipbadfiles=True,
)

# channels to compare (each is plotted in its own column, never on the same axes)
#channels = ["baseNoLjNoLjsource", "baseNoLj"]  # without and with the LJ-source cuts
channels = ["base", "base_sr"]
p = sidm_processor.SidmProcessor(
    channels,
    ["lepton_recoEff_perLepton_lxy"],
    unweighted_hist=True,  # raw counts -> exact binomial uncertainties
)

In [ ]:
output = runner.run(fileset, treename="Events", processor_instance=p)
out = output["out"]

In [ ]:
#Or, load already processed data
import coffea.util
output = coffea.util.load('output_leptonRecoEffPerLepton_vs_lxy.coffea')
out = output["out"]

Helper functions to compute the efficiency with a Clopper-Pearson interval and to plot it vs $L_{xy}$ for all four object types.

In [ ]:
# reco object -> (plot label, numerator hist, denominator hist, color).
#   denominator: parent-A Lxy of every gen lepton of the matching flavor (gen e for electrons and
#                photons, gen mu for PF and DSA muons)
#   numerator:   parent-A Lxy of the gen leptons matched one-to-one to a reco object of this type
# Colors are fixed so each object type looks the same in every panel.
reco_objs = {
    "electron": ("Electron", "electron_matchedGenE_genA_lxy", "genEs_fromA_lxy_recoEff",  "tab:blue"),
    "photon":   ("Photon",   "photon_matchedGenE_genA_lxy",   "genEs_fromA_lxy_recoEff",  "tab:orange"),
    "muon":     ("PF muon",  "muon_matchedGenMu_genA_lxy",    "genMus_fromA_lxy_recoEff", "tab:red"),
    "dsaMuon":  ("DSA muon", "dsaMuon_matchedGenMu_genA_lxy", "genMus_fromA_lxy_recoEff", "tab:purple"),
}


def clopper_pearson(k, n, cl=0.682689):
    # Clopper-Pearson interval for k passing out of n trials (k, n may be non-integer effective counts)
    k, n = np.asarray(k, dtype=float), np.asarray(n, dtype=float)
    alpha = 1 - cl
    with np.errstate(divide="ignore", invalid="ignore"):
        lo = np.where(k > 0, stats.beta.ppf(alpha/2, k, n - k + 1), 0.0)
        hi = np.where(k < n, stats.beta.ppf(1 - alpha/2, k + 1, n - k), 1.0)
    return lo, hi


def efficiency(num, den, cl=0.682689):
    # Per-bin efficiency num/den and its asymmetric (down, up) errors.
    #   den: parent-A Lxy of every gen lepton
    #   num: parent-A Lxy of the gen leptons matched to a reco object (a subset of den)
    # Bins with an empty denominator are nan. The interval uses effective counts
    # sum(w)^2/sum(w^2), which equal the raw counts for unweighted hists and keep weighted hists
    # usable as well.
    n, k, n_var = den.values(), num.values(), den.variances()
    with np.errstate(divide="ignore", invalid="ignore"):
        eff = np.where(n > 0, k/n, np.nan)
        n_eff = np.where(n_var > 0, n**2/n_var, 0.0)
    k_eff = np.clip(eff, 0, 1)*n_eff
    lo, hi = clopper_pearson(k_eff, n_eff, cl)
    return eff, eff - lo, hi - eff


def eff_hist(num, den):
    # Return an efficiency hist (variance = symmetrized error^2) and the asymmetric errors to plot
    eff, err_down, err_up = efficiency(num, den)
    h = hist.Hist(*den.axes, storage=hist.storage.Weight())
    h.view().value[:] = eff
    h.view().variance[:] = (0.5*(err_down + err_up))**2
    return h, np.array([err_down, err_up])

In [ ]:
def plot_eff_panel(ax, sample_hists, channel, cms_label=True):
    # Per-lepton efficiency of all four object types vs Lxy for ONE sample and ONE channel.
    plt.sca(ax)
    for i, (obj, (label, num_name, den_name, color)) in enumerate(reco_objs.items()):
        num = sample_hists[num_name][channel, :]
        den = sample_hists[den_name][channel, :]
        h_eff, errors = eff_hist(num, den)
        utilities.plot(h_eff, histtype="errorbar", yerr=np.nan_to_num(errors), flow="none",
                       skip_label=(i > 0 or not cms_label), label=label, color=color, markersize=12)
    ax.legend(title=f"{channel}\n" + r"gen lepton matched within $\Delta R<0.2$",
              alignment="left", loc="best")
    return den.axes[0].edges


def plot_eff_vs_lxy(proc_output, sample, channels, logx=True):
    # One figure per sample (samples are never summed). Channels are compared side by side instead
    # of on the same axes: each column is one channel, and all panels share the same x and y axes,
    # so a point in one column can be compared directly with the same point in the neighboring
    # column; horizontal grid lines help with this.
    ncol = len(channels)
    fig, axes = plt.subplots(1, ncol, figsize=(13*ncol, 11), sharex=True, sharey=True, squeeze=False)
    for col, channel in enumerate(channels):
        ax = axes[0, col]
        edges = plot_eff_panel(ax, proc_output[sample]["hists"], channel)
        ax.grid(axis="y", alpha=0.4)
        ax.set_ylim(0, 1.2)
        if logx:
            ax.set_xscale("log")
            ax.set_xlim(0.3*edges[1], edges[-1])
        ax.set_ylabel("Per-lepton reco efficiency" if col == 0 else None)
    fig.suptitle(sample_title(sample), y=1.0)
    fig.subplots_adjust(wspace=0.04, top=0.87)
    #plt.savefig(f"{sidm_path}/plots/lepton_recoEffPerLepton_vs_lxy_{sample}.png", bbox_inches="tight")
    return fig


def sample_title(sample):
    # "2Mu2E_100GeV_0p25GeV_20p0mm" -> readable title
    _, bs_mass, dp_mass, ctau = sample.split("_")
    bs_mass = bs_mass.replace("GeV", " GeV")
    dp_mass = dp_mass.replace("p", ".").replace("GeV", " GeV")
    ctau = ctau.replace("p", ".").replace("mm", " mm")
    return f"Bound state mass = {bs_mass}, dark photon mass = {dp_mass}, " + rf"$c\tau$ = {ctau}"

Per-lepton efficiency vs $L_{xy}$, one figure per sample, one column per channel.

In [ ]:
for sample in samples:
    plot_eff_vs_lxy(out, sample, channels)

In [ ]:
# save output
import coffea.util
coffea.util.save(output, "output_leptonRecoEffPerLepton_vs_lxy.coffea")